In [1]:
# frost_security_estimator.sage
#
# Security-estimation script for MAMBA-Frost.
#
# Run:
#   sage frost_security_estimator.sage
#
# This script estimates LWE-style normal-form anchors for Frost.
#
# It does not compute DFR.
# DFR should be computed with the exact discrete distribution of the total decryption error.
#
# This version performs real estimator reruns under explicit reduction-cost models:
#   MATZOV            = estimator.reduction.MATZOV
#   CoreSVP classical = ADPS16(mode="classical")
#   CoreSVP quantum   = ADPS16(mode="quantum")
#
# All reported entries are obtained by passing red_cost_model directly
# to LWE.estimate or LWE.estimate.rough.
# They are not post-hoc conversions from a single BKZ block size.

import os
import sys
import re
import math
from collections import OrderedDict


# ============================================================
# User configuration
# ============================================================

ESTIMATE_MODE = "full"
# Options:
#   "rough"  uses LWE.estimate.rough
#   "full"   uses LWE.estimate
#   "both"   runs rough and full

WIDTH_MODEL = "variance"
# Options:
#   "variance"       uses sqrt(Var(chi_{q,p}))
#   "second_moment"  uses sqrt(E[e^2])
#
# For the tie-up power-of-two quantization error, the mean is 1/2.
# The estimator input should use a centered width, so "variance" is the default.

RUN_PK_ANCHOR = True
RUN_U_ANCHOR = True
RUN_STACKED_ROW_AVERAGE = False
RUN_ATTACKER_FRIENDLY_MIN_ESTIMATE = False

APPLY_MULTI_TARGET_ELL_ADJUSTMENT = False
# If True, subtract log2(ell) from selected per-column costs.
# The paper table usually reports the per-target estimator cost, so the default is False.

PRINT_ALL_ATTACKS = True
PRINT_COST_MODEL_OBJECTS = False

ESTIMATOR_PATH_CANDIDATES = [
    os.environ.get("LATTICE_ESTIMATOR_PATH", ""),
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    "/home/dgs/lattice-estimator",
    "/home/dgs/Desktop/lattice-estimator",
    "/home/dgs/Desktop/Lattice estimator/lattice-estimator",
    "/home/dgs/Desktop/Lattice estimator/lattice-estimator/notebook",
]


# ============================================================
# Parameter sets
# ============================================================

PARAMS = OrderedDict([
    (
        "Frost-512", dict(
        level=512,
        n=2600,
        m=2600,
        ell_r=8,
        ell_s=16,
        q=2**16,
        p_pk=2**14,
        p_u=2**14,
        p_v=2**7,
        eta_s=1,
        eta_r=1,
        b_msg=4,
    )),
    (
        "Frost-384", dict(
        level=384,
        n=1928,
        m=1928,
        ell_r=8,
        ell_s=12,
        q=2**16,
        p_pk=2**13,
        p_u=2**13,
        p_v=2**9,
        eta_s=1,
        eta_r=1,
        b_msg=4,
    )),
    (
        "Frost-256", dict(
        level=256,
        n=1288,
        m=1288,
        ell=8,
        q=2**16,
        p_pk=2**13,
        p_u=2**12,
        p_v=2**8,
        eta_s=1,
        eta_r=1,
        b_msg=4,
    )),
        ("Frost-192", dict(
        level=192,
        n=880,
        m=880,
        ell=8,
        q=2**16,
        p_pk=2**11,
        p_u=2**11,
        p_v=2**6,
        eta_s=1,
        eta_r=1,
        b_msg=3,
    )),
        ("Frost-128", dict(
        level=128,
        n=512,
        m=512,
        ell=8,
        q=2**15,
        p_pk=2**10,
        p_u=2**10,
        p_v=2**5,
        eta_s=2,
        eta_r=2,
        b_msg=2,
    )),
])


# ============================================================
# Import lattice-estimator
# ============================================================

def add_estimator_paths():
    for path in ESTIMATOR_PATH_CANDIDATES:
        if not path:
            continue
        path = os.path.abspath(path)
        if os.path.isdir(path) and path not in sys.path:
            sys.path.insert(0, path)


add_estimator_paths()

try:
    from estimator import *
except Exception as ex:
    print("ERROR: could not import lattice-estimator.")
    print("Set LATTICE_ESTIMATOR_PATH or edit ESTIMATOR_PATH_CANDIDATES.")
    print("Import error:", repr(ex))
    raise

try:
    from estimator.reduction import ADPS16
except Exception as ex:
    ADPS16 = None
    print("ERROR: could not import ADPS16 from estimator.reduction.")
    print("This script requires a lattice-estimator checkout exposing ADPS16(mode=...).")
    print("Import error:", repr(ex))
    raise

try:
    from estimator.reduction import MATZOV
except Exception:
    MATZOV = None


# ============================================================
# Explicit reduction-cost models
# ============================================================

def instantiate_reduction_model(obj):
    if isinstance(obj, type):
        return obj()
    return obj


def make_matzov_cost_model():
    errors = []

    if MATZOV is not None:
        try:
            return instantiate_reduction_model(MATZOV)
        except Exception as ex:
            errors.append("MATZOV failed: {}".format(repr(ex)))

    if "RC" in globals() and hasattr(RC, "MATZOV"):
        try:
            return instantiate_reduction_model(RC.MATZOV)
        except Exception as ex:
            errors.append("RC.MATZOV failed: {}".format(repr(ex)))

    raise RuntimeError(
        "Could not construct MATZOV reduction-cost model. "
        "Use a lattice-estimator checkout exposing estimator.reduction.MATZOV or RC.MATZOV. "
        "Errors: {}".format(" | ".join(errors))
    )


def make_adps16_cost_model(mode):
    errors = []

    if ADPS16 is not None:
        try:
            return ADPS16(mode=mode)
        except Exception as ex:
            errors.append("ADPS16(mode={!r}) failed: {}".format(mode, repr(ex)))

    if "RC" in globals() and hasattr(RC, "ADPS16"):
        try:
            return RC.ADPS16(mode=mode)
        except Exception as ex:
            errors.append("RC.ADPS16(mode={!r}) failed: {}".format(mode, repr(ex)))

    raise RuntimeError(
        "Could not construct ADPS16 reduction-cost model for mode {!r}. "
        "This script requires explicit ADPS16(mode=...) support. "
        "Errors: {}".format(mode, " | ".join(errors))
    )


COST_MODELS = OrderedDict([
    ("MATZOV", dict(
        red_cost_model=make_matzov_cost_model(),
        note="MATZOV lattice-reduction cost model",
    )),
    ("CoreSVP classical", dict(
        red_cost_model=make_adps16_cost_model("classical"),
        note="ADPS16 classical Core-SVP model",
    )),
    ("CoreSVP quantum", dict(
        red_cost_model=make_adps16_cost_model("quantum"),
        note="ADPS16 quantum Core-SVP model",
    )),
])


# ============================================================
# Basic integer and size utilities
# ============================================================

def is_power_of_two(x):
    x = int(x)
    return x > 0 and (x & (x - 1)) == 0


def log2_power_of_two(x, name="value"):
    x = int(x)
    if not is_power_of_two(x):
        raise ValueError("%s must be a power of two, got %r" % (name, x))
    return int(round(math.log(x, 2)))


def ceil_log2_int(x):
    x = int(x)
    if x <= 0:
        raise ValueError("ceil_log2_int expects a positive integer")
    return int(math.ceil(math.log(x, 2)))


def exact_div(num, den, label):
    num = int(num)
    den = int(den)
    if num % den != 0:
        raise ValueError("%s is not integral: %d / %d" % (label, num, den))
    return num // den


def secret_bits_from_eta(eta):
    return ceil_log2_int(2 * int(eta) + 1)


def shape_params(P):
    if "ell_r" in P or "ell_s" in P:
        ell_r = int(P.get("ell_r", P.get("ell", 8)))
        ell_s = int(P.get("ell_s", P.get("ell", ell_r)))
    else:
        ell_r = int(P["ell"])
        ell_s = ell_r
    return ell_r, ell_s


def compute_sizes(P):
    n = int(P["n"])
    m = int(P["m"])
    ell_r, ell_s = shape_params(P)

    t_pk = log2_power_of_two(P["p_pk"], "p_pk")
    t_u = log2_power_of_two(P["p_u"], "p_u")
    t_v = log2_power_of_two(P["p_v"], "p_v")

    sk_coeff_bits = secret_bits_from_eta(P["eta_s"])

    seed_bytes_pk = 32
    seed_bytes_ct = 32
    kem_sk_extra_bytes = 64

    pk_body_bits = m * ell_r * t_pk
    ct_u_bits = n * ell_s * t_u
    ct_v_bits = ell_r * ell_s * t_v
    sk_pke_bits = n * ell_r * sk_coeff_bits

    pk_bytes = seed_bytes_pk + exact_div(pk_body_bits, 8, "pk body")
    ct_bytes = seed_bytes_ct + exact_div(ct_u_bits, 8, "ct U body") + exact_div(ct_v_bits, 8, "ct V body")
    sk_pke_bytes = exact_div(sk_pke_bits, 8, "PKE secret body")
    sk_kem_bytes = sk_pke_bytes + pk_bytes + kem_sk_extra_bytes
    ss_bytes = exact_div(int(P["level"]), 8, "shared-secret length")

    return dict(
        t_pk=t_pk,
        t_u=t_u,
        t_v=t_v,
        sk_coeff_bits=sk_coeff_bits,
        pk_bytes=pk_bytes,
        ct_bytes=ct_bytes,
        sk_pke_bytes=sk_pke_bytes,
        sk_kem_bytes=sk_kem_bytes,
        ss_bytes=ss_bytes,
        pk_ct_bytes=pk_bytes + ct_bytes,
        fo_plaintext_bits=int(P["b_msg"]) * ell_r * ell_s,
    )


# ============================================================
# Distribution utilities
# ============================================================

def cbd_variance(eta):
    return float(eta) / 2.0


def qerr_delta(q, p):
    q = int(q)
    p = int(p)
    if q % p != 0:
        raise ValueError("Frost script expects p | q, got q=%d, p=%d" % (q, p))
    return q // p


def qerr_mean(delta):
    return 0.5


def qerr_variance(delta):
    delta = float(delta)
    return (delta * delta - 1.0) / 12.0


def qerr_second_moment(delta):
    delta = float(delta)
    return (delta * delta + 2.0) / 12.0


def qerr_width(delta):
    if WIDTH_MODEL == "variance":
        return math.sqrt(qerr_variance(delta))
    if WIDTH_MODEL == "second_moment":
        return math.sqrt(qerr_second_moment(delta))
    raise ValueError("Unknown WIDTH_MODEL: %s" % WIDTH_MODEL)


def qerr_support(delta):
    delta = int(delta)
    if delta % 2 != 0:
        raise ValueError("Frost quantization error support expects even delta")
    h = delta // 2
    return -h + 1, h


def secret_width_cbd(eta):
    return math.sqrt(cbd_variance(eta))


def gaussian_distribution(sigma):
    sigma = float(sigma)

    constructors = [
        lambda: ND.DiscreteGaussian(stddev=sigma),
        lambda: ND.DiscreteGaussian(sigma=sigma),
        lambda: ND.DiscreteGaussian(sigma),
    ]

    last_ex = None
    for cons in constructors:
        try:
            return cons()
        except Exception as ex:
            last_ex = ex

    raise RuntimeError("Could not construct ND.DiscreteGaussian for sigma=%r: %r" % (sigma, last_ex))


def uniform_qerr_distribution(delta):
    a, b = qerr_support(delta)
    constructors = [
        lambda: ND.Uniform(a, b),
        lambda: ND.Uniform(lb=a, ub=b),
    ]

    last_ex = None
    for cons in constructors:
        try:
            return cons()
        except Exception as ex:
            last_ex = ex

    sigma = qerr_width(delta)
    print("WARNING: ND.Uniform(%s,%s) is unavailable; falling back to variance-matched Gaussian." % (a, b))
    return gaussian_distribution(sigma)


def cbd_distribution(eta):
    constructors = [
        lambda: ND.CenteredBinomial(eta),
        lambda: ND.CenteredBinomial(k=eta),
    ]

    last_ex = None
    for cons in constructors:
        try:
            return cons()
        except Exception as ex:
            last_ex = ex

    print("WARNING: ND.CenteredBinomial(%s) is unavailable." % eta)
    print("WARNING: falling back to a Gaussian secret with matching variance.")
    return gaussian_distribution(secret_width_cbd(eta))


def make_lwe_params(n, q, Xs, Xe, m, tag):
    constructors = [
        lambda: LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag),
        lambda: LWE.Parameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag),
    ]

    last_ex = None
    for cons in constructors:
        try:
            return cons()
        except Exception as ex:
            last_ex = ex

    raise RuntimeError("Could not construct LWE parameters for %s: %r" % (tag, last_ex))


# ============================================================
# Estimator utilities
# ============================================================

ROP_RE = re.compile(r"rop:\s*(?:[^2]*?)2\^([0-9.+\-]+)")


def run_estimator(params, red_cost_model):
    if red_cost_model is None:
        raise ValueError("red_cost_model must be provided explicitly.")

    if ESTIMATE_MODE == "rough":
        return OrderedDict([("rough", LWE.estimate.rough(params, red_cost_model=red_cost_model))])

    if ESTIMATE_MODE == "full":
        return OrderedDict([("full", LWE.estimate(params, red_cost_model=red_cost_model))])

    if ESTIMATE_MODE == "both":
        out = OrderedDict()
        out["rough"] = LWE.estimate.rough(params, red_cost_model=red_cost_model)
        out["full"] = LWE.estimate(params, red_cost_model=red_cost_model)
        return out

    raise ValueError("Unknown ESTIMATE_MODE: %s" % ESTIMATE_MODE)


def log2_number(x):
    try:
        return float(log(RR(x), 2))
    except Exception:
        return math.log(float(x), 2.0)


def parse_rop_bits_from_string(s):
    m = ROP_RE.search(str(s))
    if not m:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None


def try_get_metric(record, key):
    if record is None:
        return None

    try:
        return record[key]
    except Exception:
        pass

    try:
        getter = getattr(record, "get")
        return getter(key)
    except Exception:
        pass

    try:
        return getattr(record, key)
    except Exception:
        pass

    try:
        data = getattr(record, "_data")
        if key in data:
            return data[key]
    except Exception:
        pass

    try:
        data = getattr(record, "__dict__")
        if key in data:
            return data[key]
    except Exception:
        pass

    return None


def metric_value_to_bits(value):
    if value is None:
        return None

    parsed = parse_rop_bits_from_string(value)
    if parsed is not None:
        return parsed

    try:
        return log2_number(value)
    except Exception:
        return None


def record_rop_bits(record):
    value = try_get_metric(record, "rop")
    bits = metric_value_to_bits(value)
    if bits is not None:
        return bits

    return parse_rop_bits_from_string(record)


def best_attack_bits(result):
    best_name = None
    best_bits = None

    for name, record in result.items():
        bits = record_rop_bits(record)
        if bits is None:
            continue
        if best_bits is None or bits < best_bits:
            best_bits = bits
            best_name = str(name)

    return best_name, best_bits


def print_attack_result(result, indent="  "):
    best_name, best_bits = best_attack_bits(result)

    if PRINT_ALL_ATTACKS:
        for name, record in result.items():
            bits = record_rop_bits(record)
            if bits is None:
                print("%s%-28s no rop field" % (indent, str(name)))
                continue
            print("%s%-28s rop = 2^%.2f" % (indent, str(name), bits))

    if best_bits is None:
        print("%sselected: unavailable" % indent)
    else:
        print("%sselected: %s, 2^%.2f" % (indent, best_name, best_bits))

    return best_name, best_bits


def fmt_bits(bits):
    return "NA" if bits is None else "%.2f" % float(bits)


def fmt_pow2(x):
    try:
        if is_power_of_two(x):
            return "2^%d" % log2_power_of_two(x)
    except Exception:
        pass
    return str(x)


def print_table(title, headers, rows, indent=""):
    print("")
    if title:
        print(indent + title)

    rows = [[str(c) for c in row] for row in rows]
    headers = [str(h) for h in headers]

    if not rows:
        print(indent + "  <empty>")
        return

    widths = []
    for j, h in enumerate(headers):
        width = len(h)
        for row in rows:
            if j < len(row):
                width = max(width, len(row[j]))
        widths.append(width)

    def render(row):
        return indent + "  " + "  ".join(row[j].ljust(widths[j]) for j in range(len(headers)))

    print(render(headers))
    print(indent + "  " + "  ".join("-" * w for w in widths))
    for row in rows:
        print(render(row))


def best_record_for_model(selected_bits, cost_model_name, ell_adjustment):
    flat = []

    for surface_name, out in selected_bits.items():
        if out is None:
            continue
        if cost_model_name not in out:
            continue
        if out[cost_model_name] is None:
            continue

        for mode, rec in out[cost_model_name].items():
            bits = rec["bits"]
            if bits is None:
                continue

            adjusted_bits = bits - ell_adjustment
            flat.append((adjusted_bits, surface_name, mode, rec["best_attack"]))

    if not flat:
        return None

    flat.sort(key=lambda x: x[0])
    return flat[0]


def estimate_surface(tag, lwe_n, q, m_samples, eta_secret, sigma_error, Xe=None, xe_model="variance-matched Gaussian"):
    Xs = cbd_distribution(eta_secret)
    if Xe is None:
        Xe = gaussian_distribution(sigma_error)
    params = make_lwe_params(n=lwe_n, q=q, Xs=Xs, Xe=Xe, m=m_samples, tag=tag)

    print("")
    print("  Surface:", tag)
    print("    LWE secret dim  =", lwe_n)
    print("    LWE samples     =", m_samples)
    print("    q               =", q)
    print("    eta_secret      =", eta_secret)
    print("    sigma_error     = %.8f" % float(sigma_error))
    print("    Xe model        =", xe_model)
    print("    Xe              =", repr(Xe))

    selected = OrderedDict()

    for cost_model_name, cost_model in COST_MODELS.items():
        print("    model:", cost_model_name)
        print("      note =", cost_model["note"])
        if PRINT_COST_MODEL_OBJECTS:
            print("      red_cost_model =", repr(cost_model["red_cost_model"]))

        try:
            results_by_mode = run_estimator(params, red_cost_model=cost_model["red_cost_model"])
        except Exception as ex:
            print("      ERROR: estimator failed:", repr(ex))
            selected[cost_model_name] = None
            continue

        selected[cost_model_name] = OrderedDict()

        for mode, result in results_by_mode.items():
            print("      mode:", mode)
            best_name, best_bits = print_attack_result(result, indent="        ")
            selected[cost_model_name][mode] = dict(best_attack=best_name, bits=best_bits)

    return selected


# ============================================================
# Surface construction
# ============================================================

def validate_params(P):
    q = int(P["q"])
    if not is_power_of_two(q):
        raise ValueError("q must be a power of two")

    for key in ["p_pk", "p_u", "p_v"]:
        p = int(P[key])
        if not is_power_of_two(p):
            raise ValueError("%s must be a power of two in %s" % (key, P["name"]))
        if q % p != 0:
            raise ValueError("%s must divide q in %s" % (key, P["name"]))

    if P["level"] % 8 != 0:
        raise ValueError("level must be divisible by 8 for ss size")


def surface_widths(P):
    q = int(P["q"])

    delta_pk = qerr_delta(q, P["p_pk"])
    delta_u = qerr_delta(q, P["p_u"])
    delta_v = qerr_delta(q, P["p_v"])

    sigma_pk = qerr_width(delta_pk)
    sigma_u = qerr_width(delta_u)
    sigma_v = qerr_width(delta_v)

    return dict(
        delta_pk=delta_pk,
        delta_u=delta_u,
        delta_v=delta_v,
        sigma_pk=sigma_pk,
        sigma_u=sigma_u,
        sigma_v=sigma_v,
    )


def mixed_stacked_sigma(n_u_rows, ell_v_rows, sigma_u, sigma_v):
    numerator = n_u_rows * (sigma_u ** 2) + ell_v_rows * (sigma_v ** 2)
    denominator = n_u_rows + ell_v_rows
    return math.sqrt(numerator / float(denominator))


def attacker_friendly_min_sigma(sigma_u, sigma_v):
    return min(sigma_u, sigma_v)


def attacker_friendly_min_delta(widths):
    if widths["sigma_u"] <= widths["sigma_v"]:
        return widths["delta_u"]
    return widths["delta_v"]


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 72)
    print("MAMBA-Frost security estimator")
    print("=" * 72)
    print("Estimator mode : %s" % ESTIMATE_MODE)
    print("Width model    : %s" % WIDTH_MODEL)
    print("Estimator paths: %s" % [p for p in ESTIMATOR_PATH_CANDIDATES if p])

    print_table(
        "Reduction-cost models",
        ["Model", "Description"],
        [[name, cm["note"]] for name, cm in COST_MODELS.items()],
    )

    summary_rows = []
    size_rows = []
    security_by_scheme = OrderedDict()

    for name, P0 in PARAMS.items():
        P = dict(P0)
        P["name"] = name

        validate_params(P)

        n = int(P["n"])
        m = int(P["m"])
        ell_r, ell_s = shape_params(P)
        q = int(P["q"])

        sizes = compute_sizes(P)
        widths = surface_widths(P)

        print("")
        print("=" * 72)
        print("Profile %s" % name)
        print("=" * 72)

        print_table(
            "Parameter summary",
            ["Item", "Value"],
            [
                ["target", P["level"]],
                ["n", n],
                ["m", m],
                ["ell_r", ell_r],
                ["ell_s", ell_s],
                ["q", fmt_pow2(q)],
                ["eta_s", P["eta_s"]],
                ["eta_r", P["eta_r"]],
                ["b_msg", P["b_msg"]],
                ["p_pk", fmt_pow2(P["p_pk"])],
                ["p_u", fmt_pow2(P["p_u"])],
                ["p_v", fmt_pow2(P["p_v"])],
                ["t_pk", sizes["t_pk"]],
                ["t_u", sizes["t_u"]],
                ["t_v", sizes["t_v"]],
                ["FO plaintext bits", sizes["fo_plaintext_bits"]],
            ],
            indent="  ",
        )

        if sizes["fo_plaintext_bits"] != int(P["level"]):
            print("  WARNING: FO plaintext bits (%d) differ from nominal level (%d)." % (
                sizes["fo_plaintext_bits"], int(P["level"])
            ))

        print_table(
            "Size summary",
            ["pk", "ct", "pk+ct", "sk_PKE", "sk_KEM", "ss", "sk coeff bits"],
            [[
                "%d B" % sizes["pk_bytes"],
                "%d B" % sizes["ct_bytes"],
                "%d B" % sizes["pk_ct_bytes"],
                "%d B" % sizes["sk_pke_bytes"],
                "%d B" % sizes["sk_kem_bytes"],
                "%d B" % sizes["ss_bytes"],
                sizes["sk_coeff_bits"],
            ]],
            indent="  ",
        )

        print_table(
            "Quantization-error widths",
            ["Component", "Delta", "Mean", "Var", "E2", "Width"],
            [
                ["pk", widths["delta_pk"], "%.1f" % qerr_mean(widths["delta_pk"]), "%.8f" % qerr_variance(widths["delta_pk"]), "%.8f" % qerr_second_moment(widths["delta_pk"]), "%.8f" % widths["sigma_pk"]],
                ["u",  widths["delta_u"],  "%.1f" % qerr_mean(widths["delta_u"]),  "%.8f" % qerr_variance(widths["delta_u"]),  "%.8f" % qerr_second_moment(widths["delta_u"]),  "%.8f" % widths["sigma_u"]],
                ["v",  widths["delta_v"],  "%.1f" % qerr_mean(widths["delta_v"]),  "%.8f" % qerr_variance(widths["delta_v"]),  "%.8f" % qerr_second_moment(widths["delta_v"]),  "%.8f" % widths["sigma_v"]],
            ],
            indent="  ",
        )

        selected_bits = OrderedDict()

        if RUN_PK_ANCHOR:
            tag = "%s-pk-normal" % name
            out = estimate_surface(
                tag=tag,
                lwe_n=n,
                q=q,
                m_samples=m,
                eta_secret=P["eta_s"],
                sigma_error=widths["sigma_pk"],
                Xe=uniform_qerr_distribution(widths["delta_pk"]),
                xe_model="exact finite uniform qerr chi_{q,p_pk}",
            )
            selected_bits["pk"] = out

        if RUN_U_ANCHOR:
            tag = "%s-u-normal" % name
            out = estimate_surface(
                tag=tag,
                lwe_n=m,
                q=q,
                m_samples=n,
                eta_secret=P["eta_r"],
                sigma_error=widths["sigma_u"],
                Xe=uniform_qerr_distribution(widths["delta_u"]),
                xe_model="exact finite uniform qerr chi_{q,p_u}",
            )
            selected_bits["u"] = out

        if RUN_STACKED_ROW_AVERAGE:
            sigma_stacked = mixed_stacked_sigma(
                n_u_rows=n,
                ell_v_rows=ell_r,
                sigma_u=widths["sigma_u"],
                sigma_v=widths["sigma_v"],
            )
            tag = "%s-ct-stacked-row-average" % name
            out = estimate_surface(
                tag=tag,
                lwe_n=m,
                q=q,
                m_samples=n + ell_r,
                eta_secret=P["eta_r"],
                sigma_error=sigma_stacked,
                Xe=gaussian_distribution(sigma_stacked),
                xe_model="single-noise estimate with row-averaged variance for row-wise mixed qerr",
            )
            selected_bits["ct_stacked"] = out

        if RUN_ATTACKER_FRIENDLY_MIN_ESTIMATE:
            sigma_min = attacker_friendly_min_sigma(widths["sigma_u"], widths["sigma_v"])
            delta_min = attacker_friendly_min_delta(widths)
            tag = "%s-ct-minimum-noise-stress-estimate" % name
            out = estimate_surface(
                tag=tag,
                lwe_n=m,
                q=q,
                m_samples=n + ell_r,
                eta_secret=P["eta_r"],
                sigma_error=sigma_min,
                Xe=uniform_qerr_distribution(delta_min),
                xe_model="optional minimum-noise stress estimate using smaller stacked noise",
            )
            selected_bits["ct_min"] = out

        ell_adjustment = math.log(ell_s, 2.0) if APPLY_MULTI_TARGET_ELL_ADJUSTMENT else 0.0
        selected_model_rows = []
        security_by_scheme[name] = OrderedDict()

        for cost_model_name in COST_MODELS.keys():
            best = best_record_for_model(selected_bits, cost_model_name, ell_adjustment)

            if best is not None:
                best_bits, best_surface, best_mode, best_attack = best
                selected_model_rows.append([
                    cost_model_name,
                    best_surface,
                    best_mode,
                    best_attack,
                    fmt_bits(best_bits),
                ])
                security_by_scheme[name][cost_model_name] = best_bits
                summary_rows.append((
                    name,
                    P["level"],
                    n,
                    m,
                    ell_r,
                    ell_s,
                    q,
                    P["p_pk"],
                    P["p_u"],
                    P["p_v"],
                    sizes["pk_bytes"],
                    sizes["ct_bytes"],
                    sizes["sk_kem_bytes"],
                    sizes["ss_bytes"],
                    cost_model_name,
                    best_surface,
                    best_mode,
                    best_attack,
                    best_bits,
                ))
            else:
                selected_model_rows.append([cost_model_name, "NA", "NA", "NA", "NA"])
                security_by_scheme[name][cost_model_name] = None
                summary_rows.append((
                    name,
                    P["level"],
                    n,
                    m,
                    ell_r,
                    ell_s,
                    q,
                    P["p_pk"],
                    P["p_u"],
                    P["p_v"],
                    sizes["pk_bytes"],
                    sizes["ct_bytes"],
                    sizes["sk_kem_bytes"],
                    sizes["ss_bytes"],
                    cost_model_name,
                    "NA",
                    "NA",
                    "NA",
                    None,
                ))

        print_table(
            "Selected security records",
            ["Model", "Surface", "Mode", "Attack", "Bits"],
            selected_model_rows,
            indent="  ",
        )

        size_rows.append([
            name,
            P["level"],
            n,
            m,
            ell_r,
            ell_s,
            fmt_pow2(q),
            sizes["t_pk"],
            sizes["t_u"],
            sizes["t_v"],
            sizes["pk_bytes"],
            sizes["ct_bytes"],
            sizes["pk_ct_bytes"],
            sizes["sk_kem_bytes"],
            sizes["ss_bytes"],
        ])

    print("")
    print("=" * 72)
    print("Final summary")
    print("=" * 72)

    print_table(
        "Sizes and parameters",
        ["Scheme", "Target", "n", "m", "ell_r", "ell_s", "q", "t_pk", "t_u", "t_v", "pk", "ct", "pk+ct", "sk", "ss"],
        size_rows,
    )

    security_rows = []
    for row in size_rows:
        scheme = row[0]
        target = row[1]
        sec = security_by_scheme.get(scheme, OrderedDict())
        security_rows.append([
            scheme,
            target,
            fmt_bits(sec.get("MATZOV")),
            fmt_bits(sec.get("CoreSVP classical")),
            fmt_bits(sec.get("CoreSVP quantum")),
        ])

    print_table(
        "Selected security bits",
        ["Scheme", "Target", "MATZOV", "CoreSVP classical", "CoreSVP quantum"],
        security_rows,
    )

    detail_rows = []
    for row in summary_rows:
        (
            scheme,
            target,
            n,
            m,
            ell_r,
            ell_s,
            q,
            p_pk,
            p_u,
            p_v,
            pk,
            ct,
            sk,
            ss,
            cost_model,
            surface,
            mode,
            attack,
            bits,
        ) = row
        detail_rows.append([
            scheme,
            cost_model,
            surface,
            mode,
            attack,
            fmt_bits(bits),
        ])

    print_table(
        "Selected security details",
        ["Scheme", "Model", "Surface", "Mode", "Attack", "Bits"],
        detail_rows,
    )

    print("=" * 72)


if __name__ == "__main__":
    main()

MAMBA-Frost security estimator
Estimator mode : full
Width model    : variance
Estimator paths: ['/home/ubuntu/桌面/NGCC/Frost/LatticeEstimator/notebook', '/home/ubuntu/桌面/NGCC/Frost/LatticeEstimator', '/home/dgs/lattice-estimator', '/home/dgs/Desktop/lattice-estimator', '/home/dgs/Desktop/Lattice estimator/lattice-estimator', '/home/dgs/Desktop/Lattice estimator/lattice-estimator/notebook']

Reduction-cost models
  Model              Description                        
  -----------------  -----------------------------------
  MATZOV             MATZOV lattice-reduction cost model
  CoreSVP classical  ADPS16 classical Core-SVP model    
  CoreSVP quantum    ADPS16 quantum Core-SVP model      

Profile Frost-512

  Parameter summary
    Item               Value
    -----------------  -----
    target             512  
    n                  2600 
    m                  2600 
    ell_r              8    
    ell_s              16   
    q                  2^16 
    eta_s              1   